# teacher_hit_rate:换打分或换选点方式,能否提高 dino.txt 的指中率

**背景**:串联测试(`slot_chain_check`)里,dino.txt 按名字选出的 patch 有 62% 穿过物体的 3D 框。指中时位置误差 0.41 m,没指中时 1.78 m,所以指中率决定物体槽的质量。这里不训练任何东西,只换打分和选点方式。

**数据**:与串联测试相同。EmbodiedScan 验证集 ScanNet,229 个场景 × 3 个种子,每个样本 8 张带位姿的图,场景内唯一的物体。指中率只统计中心在视野内的物体(4949 个)。

**比较的组合**(脚本 `jepa3d/tools/improve_teacher_grounding.py`)

| 维度 | 取值 |
|---|---|
| 分辨率 | `224`:224×308 的相似度图(原做法);`ms`:再加一张 448×616 的图,池化到同一网格后取平均 |
| 打分 | `centered`:减去该 patch 对全部类别名的平均相似度(原做法);`zscore`:再除以标准差;`softmax`:该名字在全部类别名里的概率(温度 0.02) |
| 选点 | `argmax`:8 个视角里分数最高的 patch(原做法);`smooth3` / `smooth5`:先在每个视角内做 3×3 / 5×5 平均;`epipolar`:每个视角出 2 个候选,用其他视角里靠近这条射线的高分 patch 加分;`consensus`:候选先用点提示头抬到 3D,选周围候选最多的那个;`consensus+`:取获胜候选周围候选位置的加权平均 |

每个被选中的 patch 都再过一遍点提示头得到 3D 位置,所以也有位置误差和方向指标。`224 / centered / argmax` 就是串联测试的做法。方括号是按场景重采样的 95% 置信区间。

In [1]:
import json
import os
import numpy as np
import pandas as pd
pd.set_option("display.width", 220)

def load(path):
    return json.load(open(path)) if os.path.exists(path) else None

def pct(m, key):
    return f'{100 * m[key]["value"]:.1f}' if key in m else ""

def pct_ci(m, key):
    return f'{100 * m[key]["value"]:.1f} [{100 * m[key]["ci95"][0]:.1f}, {100 * m[key]["ci95"][1]:.1f}]' if key in m else ""

def metres(m, key, ci=False):
    if key not in m:
        return ""
    v = m[key]
    return f'{v["value"]:.2f} [{v["ci95"][0]:.2f}, {v["ci95"][1]:.2f}]' if ci else f'{v["value"]:.2f}'

R = "../results/"
run = load(R + "improve.json")
m = run["metrics"]
SCALES, SCORES = ("224", "ms"), ("centered", "zscore", "softmax")
SELECTS = ("argmax", "smooth3", "smooth5", "epipolar", "consensus", "consensus+")

def grid(key, fmt):
    rows = []
    for scale in SCALES:
        for score in SCORES:
            rows.append({"分辨率": scale, "打分": score, **{s: fmt(m, f"{scale}/{score}/{s}/{key}") for s in SELECTS}})
    return pd.DataFrame(rows)

print("中心在视野内的物体:", m["gt/in_view/position_m"]["n"])
print("原做法 224/centered/argmax 指中率:", pct_ci(m, "224/centered/argmax/in_view/in_box_acc"))

中心在视野内的物体: 4949
原做法 224/centered/argmax 指中率: 62.2 [60.6, 64.6]


## 1. 指中率(%):所选 patch 的射线穿过物体的 3D 框

`consensus+` 输出的是位置,不是一个 patch,所以没有指中率。

In [2]:
grid("in_view/in_box_acc", pct).drop(columns="consensus+")

,分辨率,打分,argmax,smooth3,smooth5,epipolar,consensus
0,224,centered,62.2,63.8,59.4,63.3,58.6
1,224,zscore,63.2,63.7,59.2,63.4,58.6
2,224,softmax,63.1,65.3,60.9,63.2,63.7
3,ms,centered,65.7,64.5,59.6,65.5,60.7
4,ms,zscore,66.0,64.9,59.6,65.4,59.9
5,ms,softmax,66.7,66.1,61.3,66.6,65.9


## 2. 位置误差中位数(m),可见物体

对照:类别→位置 probe(训练过)0.95 m;物体中心的真实投影 0.27 m;框内随机 patch 0.47 m。

In [3]:
grid("visible/position_m", metres)

,分辨率,打分,argmax,smooth3,smooth5,epipolar,consensus,consensus+
0,224,centered,0.67,0.66,0.67,0.66,0.72,0.71
1,224,zscore,0.68,0.66,0.67,0.67,0.73,0.72
2,224,softmax,0.66,0.64,0.64,0.66,0.64,0.62
3,ms,centered,0.67,0.66,0.66,0.67,0.72,0.71
4,ms,zscore,0.66,0.65,0.66,0.66,0.72,0.72
5,ms,softmax,0.65,0.64,0.63,0.65,0.64,0.63


## 3. 象限准确率(%),可见物体

对照:类别→位置 probe 45.6%;真实中心 84.3%;框内随机 patch 74.4%。

In [4]:
grid("visible/quadrant_acc", pct)

,分辨率,打分,argmax,smooth3,smooth5,epipolar,consensus,consensus+
0,224,centered,50.6,50.2,49.5,49.7,46.8,47.0
1,224,zscore,50.4,51.0,49.6,49.3,46.3,46.5
2,224,softmax,50.6,52.2,51.5,50.8,50.2,50.6
3,ms,centered,51.1,50.6,49.5,50.8,47.4,47.7
4,ms,zscore,51.2,51.5,50.3,50.3,46.9,47.2
5,ms,softmax,51.9,52.1,52.4,51.5,51.2,51.5


## 4. 几个代表方案的完整数字,以及按物体大小拆分的指中率

In [5]:
PICK = ["224/centered/argmax", "224/softmax/smooth3", "ms/centered/argmax", "ms/softmax/argmax", "ms/softmax/smooth3",
        "ms/softmax/consensus", "224/centered/consensus"]
rows = []
for v in PICK:
    rows.append({"方案": v, "指中率": pct_ci(m, f"{v}/in_view/in_box_acc"),
                 "< 0.5 m": pct(m, f"{v}/in_view_size_lt0.5m/in_box_acc"), "0.5–1 m": pct(m, f"{v}/in_view_size_0.5-1m/in_box_acc"),
                 "> 1 m": pct(m, f"{v}/in_view_size_gt1m/in_box_acc"),
                 "位置 (m)": metres(m, f"{v}/visible/position_m", True), "象限": pct_ci(m, f"{v}/visible/quadrant_acc")})
pd.DataFrame(rows)

,方案,指中率,< 0.5 m,0.5–1 m,> 1 m,位置 (m),象限
0,224/centered/argmax,"62.2 [60.6, 64.6]",44.2,61.8,78.0,"0.67 [0.63, 0.73]","50.6 [48.9, 52.6]"
1,224/softmax/smooth3,"65.3 [62.7, 67.6]",47.6,64.1,81.3,"0.64 [0.59, 0.68]","52.2 [50.1, 54.4]"
2,ms/centered/argmax,"65.7 [63.3, 68.0]",50.0,64.1,80.1,"0.67 [0.62, 0.72]","51.1 [49.0, 53.0]"
3,ms/softmax/argmax,"66.7 [64.2, 69.4]",50.8,64.8,81.6,"0.65 [0.60, 0.70]","51.9 [50.0, 53.7]"
4,ms/softmax/smooth3,"66.1 [63.6, 68.4]",48.6,65.2,81.7,"0.64 [0.59, 0.69]","52.1 [49.9, 53.9]"
5,ms/softmax/consensus,"65.9 [63.2, 67.9]",49.2,64.4,81.2,"0.64 [0.60, 0.69]","51.2 [49.3, 53.6]"
6,224/centered/consensus,"58.6 [56.6, 60.9]",40.9,59.6,73.2,"0.72 [0.67, 0.79]","46.8 [45.0, 48.7]"


## 5. 上限:分数图本身还有多少余地

- `AUC`:穿过框的 patch 分数高于不穿过框的 patch 的概率。
- `候选里有指中的`:每个视角 2 个候选(共 16 个)里至少有一个穿过框。这是 `epipolar` 和 `consensus` 的上限。

In [6]:
pd.DataFrame([{"分辨率": s, "打分": sc, "AUC": f'{m[f"{s}/{sc}/in_view/auc_acc"]["value"]:.3f}',
               "候选里有指中的": pct_ci(m, f"{s}/{sc}/any_cand/in_view/in_box_acc")} for s in SCALES for sc in SCORES])

,分辨率,打分,AUC,候选里有指中的
0,224,centered,0.805,"81.9 [80.0, 83.8]"
1,224,zscore,0.807,"82.7 [80.9, 84.7]"
2,224,softmax,0.807,"82.7 [81.0, 84.4]"
3,ms,centered,0.828,"82.9 [80.8, 84.8]"
4,ms,zscore,0.832,"83.5 [81.5, 85.6]"
5,ms,softmax,0.831,"83.4 [81.5, 85.2]"


## 结论

**免训练的办法只能把指中率从 62% 提到 66–67%,提升很小。**

- **有用的只有加一档高分辨率。** `ms` 比 `224` 高 3–4 个点(62.2% → 65.7%),主要来自小物体(< 0.5 m:44% → 50%)。
- **换打分几乎没差别。** `centered`、`zscore`、`softmax` 相差 1 个点以内,在置信区间里。
- **平滑没用。** 3×3 平滑持平,5×5 平滑让小物体掉到 35%(小物体只占一两个 patch,被抹掉了)。
- **跨视角印证没用。** `epipolar` 与 `argmax` 持平;`consensus` 在 `centered` 打分下反而降到 59%。
- **位置误差和方向指标基本不动。** 最好的组合 0.63 m / 52%,原做法 0.67 m / 51%。

**为什么跨视角印证不起作用(推断,未单独验证)**:16 个候选里有指中的占 82–83%,说明对的候选多数时候是有的,但选不出来。场景里唯一的物体往往只在 1–2 个视角里出现,其他视角给不出支持;而错的候选(相似的东西)可能在多个视角都出现。

**含义**
- 指中率的瓶颈在 dino.txt 的分数图本身,不在怎么从图里取点。要再提高,需要训练(比如在分数图上学一个选点头),或者换一种不依赖单点的用法(把多个候选都交给下游)。
- 每个名字给 16 个候选时,82% 的物体至少有一个候选是对的。这支持「一个名字多个候选槽」的集成方式。

**没验证的**:只测了场景内唯一的物体;温度、平滑核、σ 各只试了一个值;一个随机种子(框内随机 patch)。